In [1]:
import os
print(os.environ.get("PYTHONPATH"))
PROJECT_ROOT = os.environ.get("PYTHONPATH")

C:/Users/BISITE/Desktop/GNN_CoPiPred


In [2]:
import os
import yaml
import torch
from pathlib import Path
import subprocess
from typing import Dict, List

# Load configuration
config_path = "config/config.yaml"
with open(os.path.join(PROJECT_ROOT, config_path), 'r') as f:
    config = yaml.safe_load(f)

# Define paths
pdb_path = Path(os.path.join(PROJECT_ROOT, "data/epitope3d/pdb_structure"))
output_path = Path(os.path.join(PROJECT_ROOT, "data/epitope3d/embeddings/ESM2"))

# Create output directory if it doesn't exist
output_path.mkdir(parents=True, exist_ok=True)

# Configuration parameters
model_name = config.get('model_embedding', 'facebook/esm2_t33_650M_UR50D')
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Configuration loaded:")
print(f"  Model: {model_name}")
print(f"  Device: {device}")
print(f"  PDB Path: {pdb_path}")
print(f"  Output Path: {output_path}")


Configuration loaded:
  Model: facebook/esm2_t33_650M_UR50D
  Device: cuda
  PDB Path: C:\Users\BISITE\Desktop\GNN_CoPiPred\data\epitope3d\pdb_structure
  Output Path: C:\Users\BISITE\Desktop\GNN_CoPiPred\data\epitope3d\embeddings\ESM2


In [4]:
try:
    from transformers import AutoTokenizer, AutoModel
    print("Transformers library found")
except ImportError:
    print("Installing transformers library...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
    from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model
print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
model = model.to(device)
model.eval()
print("Model loaded successfully!")


c:\Users\BISITE\Desktop\GNN_CoPiPred\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Transformers library found
Loading model: facebook/esm2_t33_650M_UR50D


Loading weights: 100%|██████████| 566/566 [00:00<00:00, 1140.04it/s, Materializing param=encoder.layer.32.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight         | MISSING    | 
pooler.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [5]:
from Bio import PDB
import numpy as np

# Load amino acid dictionary from config
aa_dict = config.get('aminoacid_dict', {})

def extract_sequence_from_pdb(pdb_file: str) -> str:
    """Extract protein sequence from a PDB file using ATOM records"""
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('pdb', pdb_file)
    
    ppb = PDB.PPBuilder()
    peptides = ppb.build_peptides(structure)
    
    if peptides:
        # Get the first peptide chain
        seq = str(peptides[0].get_sequence())
        return seq
    return ""

# Find all PDB files
pdb_files = list(pdb_path.glob("*.pdb"))
print(f"Found {len(pdb_files)} PDB files")
if len(pdb_files) > 0:
    print(f"Sample files: {[f.name for f in pdb_files[:5]]}")


Found 245 PDB files
Sample files: ['1ALU.pdb', '1B9C.pdb', '1BR6.pdb', '1C7P.pdb', '1CFI.pdb']


In [6]:
def generate_embedding(sequence: str) -> torch.Tensor:
    """Generate ESM2 embedding for a protein sequence"""
    if not sequence:
        return None
    
    # Tokenize sequence
    input_ids = tokenizer.encode(sequence, return_tensors="pt")
    input_ids = input_ids.to(device)
    
    # Generate embeddings
    with torch.no_grad():
        output = model(input_ids, output_hidden_states=True)
        # Use the last hidden state
        embeddings = output.hidden_states[-1].squeeze(0)  # Shape: (seq_len, embedding_dim)
    
    return embeddings.cpu()

# Generate embeddings for all PDB files
processed_count = 0
failed_count = 0

for pdb_file in pdb_files:
    try:
        # Extract sequence from PDB
        sequence = extract_sequence_from_pdb(str(pdb_file))
        
        if sequence and len(sequence) > 0:
            # Generate embedding
            embedding = generate_embedding(sequence)
            
            if embedding is not None:
                # Save embedding with same name as PDB file (but .pt extension)
                output_file = output_path / f"{pdb_file.stem}.pt"
                torch.save(embedding, output_file)
                processed_count += 1
                
                if processed_count % 10 == 0:
                    print(f"Processed {processed_count} files...")
        else:
            print(f"Warning: No sequence extracted from {pdb_file.name}")
            failed_count += 1
            
    except Exception as e:
        print(f"Error processing {pdb_file.name}: {str(e)}")
        failed_count += 1

print(f"\n✓ Successfully processed: {processed_count} files")
print(f"✗ Failed: {failed_count} files")
print(f"Embeddings saved to: {output_path}")


Processed 10 files...
Processed 20 files...
Processed 30 files...
Processed 40 files...
Processed 50 files...
Processed 60 files...
Processed 70 files...
Processed 80 files...
Processed 90 files...
Processed 100 files...
Processed 110 files...
Processed 120 files...
Processed 130 files...
Processed 140 files...
Processed 150 files...
Processed 160 files...
Processed 170 files...
Processed 180 files...
Processed 190 files...
Processed 200 files...
Processed 210 files...
Processed 220 files...
Processed 230 files...
Processed 240 files...

✓ Successfully processed: 245 files
✗ Failed: 0 files
Embeddings saved to: C:\Users\BISITE\Desktop\GNN_CoPiPred\data\epitope3d\embeddings\ESM2


In [7]:
# Verify embeddings
embedding_files = list(output_path.glob("*.pt"))
print(f"\n✓ Total embeddings generated: {len(embedding_files)}")

if len(embedding_files) > 0:
    # Show statistics of first few embeddings
    print("\nSample embeddings:")
    for emb_file in embedding_files[:5]:
        emb = torch.load(emb_file)
        print(f"  {emb_file.name}: shape {emb.shape}, dtype {emb.dtype}")
    
    # Verify output directory
    total_size = sum(f.stat().st_size for f in embedding_files) / (1024**2)  # Size in MB
    print(f"\nTotal size of embeddings: {total_size:.2f} MB")
else:
    print("No embeddings were generated. Please check the PDB path and files.")



✓ Total embeddings generated: 245

Sample embeddings:
  1ALU.pt: shape torch.Size([35, 1280]), dtype torch.float32
  1B9C.pt: shape torch.Size([63, 1280]), dtype torch.float32
  1BR6.pt: shape torch.Size([270, 1280]), dtype torch.float32
  1C7P.pt: shape torch.Size([135, 1280]), dtype torch.float32
  1CFI.pt: shape torch.Size([8, 1280]), dtype torch.float32

Total size of embeddings: 219.84 MB


C:\Users\BISITE\AppData\Local\Temp\ipykernel_27680\3501094864.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  emb = torch.load(emb_file)
